# VWAP Hug → Green Lift-off — Setup Backtest & SL/Target Optimization

**Setup (Resistance Marker, intraday):** a resistance level **tested ≥ 2×**, then price **holds below** it while the
closes **hug a flat VWAP** for **≥ 5 candles**, then a **green candle closes clearly above the VWAP** (the *lift-off*).

**This notebook:**
1. Scans the whole **15m universe** (`ema_scanner/data/15m/*`) and collects every lift-off entry.
2. Backtests with **entry = next bar's open** after the lift-off candle.
3. Optimizes **SL & target** two ways and maximizes **total PnL** subject to **win-rate > 25%**:
   - **(A)** fixed **%** SL  ×  % target
   - **(B)** **structural SL = nearest trough** (the low of the hug consolidation), target = **R × risk**


In [1]:
import csv, glob
import pandas as pd

DATA = "ema_scanner/data/15m"

def load(p):
    o=[]
    with open(p) as f:
        for r in csv.DictReader(f):
            if "" in (r["open"],r["high"],r["low"],r["close"],r["volume"]): continue
            o.append({"time":int(r["timestamp"]),"o":float(r["open"]),"h":float(r["high"]),
                      "l":float(r["low"]),"c":float(r["close"]),"v":float(r["volume"])})
    return o

def vwap(c):                       # session VWAP, resets each trading day (intraday)
    o=[];pv=vv=0;cur=None
    for x in c:
        d=x["time"]//86400
        if d!=cur:pv=vv=0;cur=d
        tp=(x["h"]+x["l"]+x["c"])/3;v=x["v"];pv+=tp*v;vv+=v
        o.append(pv/vv if vv>0 else tp)
    return o


In [2]:
def cluster(piv,TOL=0.0035):       # cluster swing-high pivots into resistance zones (15m TOL=0.35%)
    piv=sorted(piv,key=lambda p:p["price"]);MT=TOL*0.25;MB=12;zs=[]
    for p in piv:
        z=zs[-1] if zs else None;pl=False
        if z:
            eff=TOL
            if z["n"]>=2:
                sr=(z["top"]-z["bottom"])/z["avg"];eff=min(TOL,max(MT,sr*1.5))
            if abs(p["price"]-z["avg"])<=z["avg"]*eff:
                if any(abs(p["i"]-q["i"])<MB for q in z["pts"]):continue
                z["sum"]+=p["price"];z["n"]+=1;z["avg"]=z["sum"]/z["n"]
                z["top"]=max(z["top"],p["price"]);z["bottom"]=min(z["bottom"],p["price"]);z["pts"].append(p);pl=True
        if not pl:zs.append({"sum":p["price"],"n":1,"avg":p["price"],"top":p["price"],"bottom":p["price"],"pts":[p]})
    return zs

def liftoffs(vis,vw,z,BREAK=0.0015,HUG=0.004,FLAT=0.004,MINHUG=5):
    # returns list of (lift_off_index, nearest_trough_price) for one resistance zone
    n=len(vis);top=z["top"];t=sorted(p["i"] for p in z["pts"])
    if len(t)<2:return []
    ft,st,lt=t[0],t[1],t[-1];brk=n
    for j in range(ft+1,n):
        if vis[j]["c"]>top*(1+BREAK):brk=j;break
    if lt>brk:return []                                   # role-reversal level -> skip
    a=st;cap=min(brk-1,n-1,a+80);br=lambda i:vis[i]["c"]<top*(1+BREAK);i=a+1;out=[]
    while i+MINHUG<=cap:
        h0=i;h1=i-1;j=i
        while j<=cap and br(j) and abs(vis[j]["c"]-vw[j])/vw[j]<=HUG:h1=j;j+=1   # hug run (closes on the line)
        if h1-h0+1>=MINHUG:
            if abs(vw[h1]-vw[h0])/vw[h0]<=FLAT and j<=cap:                       # VWAP flat across the hug
                c=vis[j]
                if c["c"]>c["o"] and c["c"]>vw[j]*(1+HUG) and br(j):             # GREEN lift-off above VWAP, below resistance
                    trough=min(vis[k]["l"] for k in range(h0,j+1))               # nearest trough = hug low
                    out.append((j,trough));i=j;continue
            i=h1+1
        else:i+=1
    return out


In [3]:
def collect():
    K=8;series=[];entries=[]
    for p in sorted(glob.glob(DATA+"/*_historical.csv")):
        vis=load(p)
        if len(vis)<60: continue
        vw=vwap(vis);piv=[]
        for i in range(K,len(vis)-K):
            if all(vis[j]["h"]<=vis[i]["h"] for j in range(i-K,i+K+1)):piv.append({"i":i,"price":vis[i]["h"]})
        evs=[]
        for z in cluster(piv):
            if z["n"]>=2: evs+=liftoffs(vis,vw,z)
        if evs:
            sidx=len(series);series.append(vis)
            for L,tr in sorted(set(evs)):
                if L+1<len(vis): entries.append((sidx,L,tr))
    return series,entries

series,entries=collect()
print(f"{len(entries)} setup entries across {len(series)} stocks")


4303 setup entries across 519 stocks


In [4]:
def sim(mode,sl=None,tgt=None,R=None,buf=0.001,MAXHOLD=60):
    # entry = next bar open; SL/target checked intrabar; ties -> SL first (conservative); else exit at market after MAXHOLD
    wins=0;gw=gl=0.0;rets=[];used=0
    for sidx,L,tr in entries:
        vis=series[sidx];E=vis[L+1]["o"]
        if mode=="pct":
            slp=E*(1-sl);risk=sl;tp=E*(1+tgt)
        else:
            slp=tr*(1-buf);risk=(E-slp)/E
            if risk<=0 or risk>0.08: continue              # trough above entry / too far -> not tradable
            tp=E*(1+R*risk)
        used+=1;r=None
        for k in range(L+1,min(L+1+MAXHOLD,len(vis))):
            c=vis[k];hs=c["l"]<=slp;ht=c["h"]>=tp
            if hs: r=-risk;break
            if ht: r=(tp-E)/E;break
        if r is None: r=(vis[min(L+MAXHOLD,len(vis)-1)]["c"]-E)/E
        rets.append(r)
        if r>0: wins+=1;gw+=r
        else: gl+=abs(r)
    n=len(rets)
    return dict(n=n,win=wins/n,pnl=sum(rets),exp=sum(rets)/n,pf=(gw/gl if gl>0 else float('inf')),used=used)


## (A) Fixed %-SL  ×  %-target grid


In [5]:
SLS=[0.0075,0.01,0.0125,0.015,0.02,0.025,0.03]
TGTS=[0.01,0.015,0.02,0.03,0.04,0.05,0.06,0.08,0.10]
rows=[]
for sl in SLS:
    for tgt in TGTS:
        r=sim("pct",sl=sl,tgt=tgt)
        rows.append(dict(sl=sl*100,tgt=tgt*100,RR=round(tgt/sl,2),win=round(r['win']*100,1),
                         pf=round(r['pf'],2),exp=round(r['exp']*100,3),totPnL=round(r['pnl']*100,0),n=r['n']))
A=pd.DataFrame(rows)
A=A[A.win>25].sort_values("totPnL",ascending=False).reset_index(drop=True)
A.head(10)


,sl,tgt,RR,win,pf,exp,totPnL,n
0,1.25,10.0,8.00,31.8,1.31,0.250,1075.0,4303
1,1.50,10.0,6.67,35.3,1.27,0.246,1059.0,4303
2,1.25,8.0,6.40,31.9,1.29,0.234,1008.0,4303
3,1.50,8.0,5.33,35.3,1.25,0.227,978.0,4303
4,1.00,10.0,10.00,27.1,1.32,0.226,973.0,4303
5,2.00,10.0,5.00,39.6,1.21,0.217,933.0,4303
6,1.00,8.0,8.00,27.2,1.30,0.212,913.0,4303
7,2.00,8.0,4.00,39.7,1.19,0.200,860.0,4303
8,1.25,6.0,4.80,32.2,1.24,0.198,852.0,4303
9,1.50,6.0,4.00,35.6,1.21,0.194,834.0,4303


## (B) Structural SL = nearest trough (hug low), target = R × risk


In [6]:
rows=[]
for R in [2,3,4,5,6,8,10,12,15,20]:
    r=sim("trough",R=R)
    rows.append(dict(RR=R,win=round(r['win']*100,1),pf=round(r['pf'],2),
                     exp=round(r['exp']*100,3),totPnL=round(r['pnl']*100,0),trades=r['used']))
B=pd.DataFrame(rows)
B=B[B.win>25].sort_values("totPnL",ascending=False).reset_index(drop=True)
B


,RR,win,pf,exp,totPnL,trades
0,20,31.8,1.37,0.302,1296.0,4293
1,15,31.8,1.36,0.292,1253.0,4293
2,12,31.9,1.35,0.286,1228.0,4293
3,10,31.9,1.33,0.271,1163.0,4293
4,8,31.9,1.32,0.261,1122.0,4293
5,6,32.2,1.29,0.237,1017.0,4293
6,5,32.4,1.25,0.199,855.0,4293
7,4,33.4,1.24,0.193,829.0,4293
8,3,35.1,1.23,0.182,783.0,4293
9,2,40.0,1.21,0.151,650.0,4293


## Result / chosen parameters

- **Structural trough SL beats fixed-% SL** on PnL, profit factor and expectancy.
- **Best:** SL = nearest trough (hug low, −0.1% buffer), **target = 10–15 × risk** →
  **win-rate ≈ 32 %**, **PF ≈ 1.33–1.36**, **expectancy ≈ +0.27–0.29 %/trade**, **total PnL ≈ +1160–1250 %** over ~4,300 trades.
- The setup is positively skewed: most exits are small (trough stop / time-out near breakeven) while a minority of winners run far,
  so a **wide target (high R) maximizes PnL** while win-rate stays comfortably above the 25 % floor.

> Conservative assumptions: entry on next-bar open, SL-first on ambiguous bars, no costs/slippage. Add brokerage/slippage for a net figure.
